In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42

HIGHLIGHTS = {
    'Erythritol': '#FF0000',
    'Xylitol': '#008000',
    'Inulin': '#800080',
    'Mannose': '#FFA500',
    'Resistant maltodextrin': '#0000FF',
    'Resistant starch': '#00CED1'
}

LEGEND_ORDER = ['Resistant maltodextrin', 'Resistant starch', 'Mannose', 'Inulin', 'Xylitol', 'Erythritol']
COLOR_SIG_OTHER = '#888888'
COLOR_NON_SIG = '#CCCCCC'

file_path = 'Propionate(mM).csv'
df = pd.read_csv(file_path)

donor_cols = [c for c in df.columns if c.startswith('HS-')]
df[donor_cols] = df[donor_cols].apply(pd.to_numeric)

ctrl_row = df[df['KULFFI'] == 'Control']
if ctrl_row.empty:
    raise ValueError("Control row not found in the dataset.")

ctrl_vals = ctrl_row[donor_cols].values[0].astype(float)

results = []
for idx, row in df.iterrows():
    mat_name = row['KULFFI']
    if mat_name == 'Control': continue

    mat_vals = row[donor_cols].values.astype(float)

    diff = mat_vals - ctrl_vals
    if np.all(diff == 0):
        p_val = 1.0
    else:
        try:
            stat, p_val = wilcoxon(mat_vals, ctrl_vals)
        except ValueError:
            p_val = 1.0

    mean_mat = np.mean(mat_vals)
    mean_ctrl = np.mean(ctrl_vals)

    epsilon = 0.01
    log2fc = np.log2((mean_mat + epsilon) / (mean_ctrl + epsilon))

    results.append({'Material': mat_name, 'p_val': p_val, 'Log2FC': log2fc})

res_df = pd.DataFrame(results)

res_df['q_val'] = multipletests(res_df['p_val'], method='fdr_bh')[1]
res_df['neg_log10_q'] = -np.log10(res_df['q_val'] + 1e-300)

fig, ax = plt.subplots(figsize=(10, 8))

non_high = res_df[~res_df['Material'].isin(HIGHLIGHTS.keys())]
sig_mask = non_high['q_val'] < 0.05

ax.scatter(non_high.loc[sig_mask, 'Log2FC'], non_high.loc[sig_mask, 'neg_log10_q'],
           c=COLOR_SIG_OTHER, alpha=0.6, s=60, edgecolor='none')
ax.scatter(non_high.loc[~sig_mask, 'Log2FC'], non_high.loc[~sig_mask, 'neg_log10_q'],
           c=COLOR_NON_SIG, alpha=0.5, s=40, edgecolor='none')

for name in LEGEND_ORDER:
    row = res_df[res_df['Material'] == name]
    if not row.empty:
        ax.scatter(row['Log2FC'], row['neg_log10_q'],
                   c=HIGHLIGHTS[name], s=200, edgecolors='black', linewidth=1.5, zorder=10)

ax.axvline(0, color='black', linestyle=':', linewidth=2.0)
ax.axhline(-np.log10(0.05), color='black', linestyle='-', linewidth=2.0)

ax.set_xlim(-1.0, 1.0)
ax.set_ylim(bottom=0)

ax.set_xlabel('Log2 Fold Change', fontsize=24, fontweight='bold', labelpad=15)
ax.set_ylabel('-Log10 q-value', fontsize=24, fontweight='bold', labelpad=15)

ax.tick_params(axis='both', labelsize=20, width=2.0, length=6)

legend_handles = []
for n in LEGEND_ORDER:
    label_name = n
    if n == 'Resistant maltodextrin': label_name = 'RMD'
    if n == 'Resistant starch': label_name = 'RS'
    legend_handles.append(mlines.Line2D([], [], color=HIGHLIGHTS[n], marker='o',
                                        linestyle='None', markersize=12,
                                        markeredgecolor='black', label=label_name))

legend_handles.append(mlines.Line2D([], [], color=COLOR_SIG_OTHER, marker='o',
                                    linestyle='None', markersize=10, alpha=0.6, label='Significant (q < 0.05)'))
legend_handles.append(mlines.Line2D([], [], color=COLOR_NON_SIG, marker='o',
                                    linestyle='None', markersize=10, alpha=0.5, label='Not Significant'))

ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=18)

for s in ['top', 'right']: ax.spines[s].set_visible(False)
for s in ['left', 'bottom']: ax.spines[s].set_linewidth(2.0)

plt.tight_layout()
plt.savefig('Figure_3b.pdf', dpi=600, bbox_inches='tight')
plt.close()